# Manuscript figures — rainfed soybean water deficit (2014–2023)

Renamed publication copy of **S24**. Generates seven figure sets from seven processed CSV files.
The numerical operations, plotting routines, output file basenames and styles are unchanged.
Only input file locations/names, output folder name, and documentation were updated.
See `docs/file_name_mapping.csv` for mapping to original names.


In [ ]:
# =============================================================================
# SCRIPT 24 — FIGURAS FINAIS (APENAS PLOTS)
# DEF_ACUM e produtividade da soja sequeiro
# Brasil, 2014–2023
# =============================================================================
#
# INSTRUÇÕES
# ------------
# 1) Este script recria APENAS as figuras finais.
#    Ele não recalcula DEF_ACUM, produtividade da soja ou balanço hídrico.
#
# 2) Execute o notebook a partir da raiz do repositório ou da pasta notebooks/.
#    A pasta raiz do projeto é detectada automaticamente na CÉLULA 2.
#
# 3) O script procura automaticamente as tabelas necessárias dentro de:
#       PASTA_PROJETO / "data" / "analysis"
#
# 4) Todas as saídas são exportadas para:
#       PASTA_PROJETO / "results" / "figures" / NOME_ANALISE
#
# 5) Cada figura é exportada em duas versões:
#       A) sem títulos dos eixos
#       B) com títulos dos eixos
#
# 6) Os títulos das figuras são omitidos propositalmente.
#
# 7) Todos os controles de estilo que geralmente precisam de revisão
#    (tamanho da fonte, tamanho dos ticks, espessura das linhas, DPI, tamanho da figura etc.)
#    estão centralizados na CÉLULA 3 para facilitar os ajustes.
#
# 8) Exportação em qualidade de publicação:
#    - PNG em 600 dpi;
#    - TIFF em 600 dpi;
#    - saída vetorial em PDF;
#    - saída vetorial em SVG com texto editável;
#    - fontes TrueType incorporadas em PDF/PS;
#    - linhas e textos com antialiasing;
#    - fundo branco e dimensões consistentes entre as figuras.
#
# 9) Para submissão em periódicos, prefira PDF/SVG sempre que o periódico aceitar
#    gráficos vetoriais. Use TIFF/PNG quando a revista solicitar arquivos raster.
#
# 10) Este script usa sete tabelas de entrada fornecidas pelo pesquisador.
#     A origem computacional dessas tabelas deve ser documentada separadamente;
#     este notebook NÃO executa o processamento que as gerou.
# =============================================================================


In [ ]:
# =============================================================================
# CÉLULA 1 — IMPORTAÇÕES
# =============================================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch


In [ ]:
# =============================================================================
# CÉLULA 2 — CONFIGURAÇÕES DO PROJETO
# =============================================================================

# Caminho relativo: executar a partir da raiz do repositório ou de notebooks/.
PASTA_PROJETO = Path.cwd().resolve()
if not (PASTA_PROJETO / "data" / "analysis").is_dir():
    PASTA_PROJETO = PASTA_PROJETO.parent
if not (PASTA_PROJETO / "data" / "analysis").is_dir():
    raise FileNotFoundError("Execute a partir da raiz do repositório ou da pasta notebooks/.")

NOME_ANALISE = "manuscript_figures"

PASTA_BASE = PASTA_PROJETO / "results"
PASTA_TABELAS_GERAL = PASTA_PROJETO / "data" / "analysis"
PASTA_FIGURAS = PASTA_BASE / "figures" / NOME_ANALISE
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

ANOS_ANALISE = list(range(2014, 2024))
ANOS_ESPERADOS = ANOS_ANALISE.copy()

# ---------------------------------------------------------------------------
# TABELAS DE ENTRADA DAS ANÁLISES ANTERIORES
# ---------------------------------------------------------------------------
# Previously processed DEF_ACUM summary tables (provenance in README)
ARQ_DEF_MACRO = "water_deficit_by_macroregion.csv"
ARQ_DEF_MICRO = "water_deficit_by_edaphoclimatic_region.csv"
ARQ_RANKING_MICRO = "water_deficit_region_ranking.csv"
ARQ_DEF_SOLOS = "water_deficit_by_soil_class.csv"
ARQ_DEF_SOLO_MACRO = "water_deficit_by_soil_and_macroregion.csv"

# Previously processed DEF_ACUM x yield summary tables (provenance in README)
ARQ_SOLO_ANUAL_DEF_PROD = "annual_water_deficit_and_yield_by_soil.csv"
ARQ_SOLO_MACRO_ANUAL_DEF_PROD = "annual_water_deficit_and_yield_by_soil_and_macroregion.csv"

# ---------------------------------------------------------------------------
# ORDEM DAS CATEGORIAS
# ---------------------------------------------------------------------------

MACROS_ORDEM = ["Macro_1", "Macro_2", "Macro_3", "Macro_4", "Macro_5"]

ORDEM_SOLOS = [
    "Neossolo Quartzarênico",
    "Latossolo textura média",
    "Latossolo argiloso",
    "Latossolo muito argiloso",
    "Argissolo",
]

# ---------------------------------------------------------------------------
# RÓTULOS DOS SOLOS PARA EXIBIÇÃO NAS FIGURAS
# ---------------------------------------------------------------------------
# Os nomes originais em português são mantidos internamente porque são os nomes
# existentes nas tabelas já processadas. Nas figuras, porém, são exibidos os
# termos adotados no manuscrito após a harmonização operacional SiBCS → WRB.
#
# Esta harmonização foi usada para estratificação analítica e não representa
# equivalência taxonômica estrita entre os sistemas SiBCS e WRB.

ROTULOS_SOLOS_INGLES = {
    "Neossolo Quartzarênico": "Arenosols",
    "Latossolo textura média": "Medium-textured Ferralsols",
    "Latossolo argiloso": "Clayey Ferralsols",
    "Latossolo muito argiloso": "Very clayey Ferralsols",
    "Argissolo": "Acrisols, Lixisols, and Alisols group",
}

# ---------------------------------------------------------------------------
# OPÇÕES DE EXPORTAÇÃO
# ---------------------------------------------------------------------------

SALVAR_PNG = True
SALVAR_TIFF = True
SALVAR_PDF = True
SALVAR_SVG = True

# 600 dpi is suitable for publication-quality line art and combined figures.
DPI = 600

GERAR_VERSAO_SEM_TITULOS_EIXOS = True
GERAR_VERSAO_COM_TITULOS_EIXOS = True

MOSTRAR_FIGURAS = True

print("Pasta do projeto:", PASTA_PROJETO)
print("Pasta raiz das tabelas:", PASTA_TABELAS_GERAL)
print("Pasta de saída das figuras:", PASTA_FIGURAS)


In [ ]:
# =============================================================================
# CÉLULA 3 — CONFIGURAÇÕES MESTRAS DE ESTILO (EDITE ESTE BLOCO COM FREQUÊNCIA)
# =============================================================================
# CONFIGURAÇÕES DE FIGURAS EM QUALIDADE DE PUBLICAÇÃO
#
# Uso recomendado:
# - Mantenha todas as figuras finais do artigo com largura física consistente.
# - Exportações vetoriais (PDF/SVG) preservam a máxima nitidez.
# - Exportações raster (PNG/TIFF) são salvas em 600 dpi.
# - Os tamanhos de fonte abaixo estão em pontos e podem ser ajustados durante a revisão do manuscrito.
#
# Larguras típicas de periódicos:
# - Uma coluna: ~85–90 mm
# - Duas colunas: ~175–185 mm
#
# As figuras finais deste script foram pensadas principalmente para largura de duas colunas.
# =============================================================================

def mm_to_inch(mm):
    """Converte milímetros em polegadas para definir dimensões físicas precisas de publicação."""
    return mm / 25.4


# -----------------------------------------------------------------------------
# LARGURA FÍSICA DAS FIGURAS
# -----------------------------------------------------------------------------

DOUBLE_COLUMN_WIDTH_MM = 183
SINGLE_COLUMN_WIDTH_MM = 89

DOUBLE_COLUMN_WIDTH = mm_to_inch(DOUBLE_COLUMN_WIDTH_MM)
SINGLE_COLUMN_WIDTH = mm_to_inch(SINGLE_COLUMN_WIDTH_MM)


# -----------------------------------------------------------------------------
# TIPOGRAFIA — EDITE COM FREQUÊNCIA DURANTE A REVISÃO DAS FIGURAS
# -----------------------------------------------------------------------------

FONT_FAMILY = "Arial"

BASE_FONT_SIZE = 12
TICK_FONT_SIZE = 12
AXIS_TITLE_FONT_SIZE = 14
LEGEND_FONT_SIZE = 12
LEGEND_TITLE_FONT_SIZE = 13
ANNOTATION_FONT_SIZE = 12

HEATMAP_CELL_FONT_SIZE = 13
ROW_LABEL_FONT_SIZE = 13
HEATMAP_CBAR_TICK_FONT_SIZE = 12
HEATMAP_CBAR_LABEL_FONT_SIZE = 14
HEATMAP_AXIS_TICK_FONT_SIZE = 13
HEATMAP_AXIS_LABEL_FONT_SIZE = 14

# Se o periódico solicitar texto maior, altere apenas estes valores.
# A useful alternative is:
# BASE_FONT_SIZE = 11
# TICK_FONT_SIZE = 11
# AXIS_TITLE_FONT_SIZE = 13


# -----------------------------------------------------------------------------
# LINHAS E EIXOS
# -----------------------------------------------------------------------------

AXIS_LINE_WIDTH = 0.9
GRID_LINE_WIDTH = 0.6
BOX_LINE_WIDTH = 1.0
MEDIAN_LINE_WIDTH = 1.3
WHISKER_LINE_WIDTH = 0.9
CAP_LINE_WIDTH = 0.9
BAR_EDGE_WIDTH = 0.7

GRID_ALPHA = 0.18
BOX_ALPHA = 0.90
BAR_ALPHA = 0.96


# -----------------------------------------------------------------------------
# ROTAÇÕES
# -----------------------------------------------------------------------------

ROTATE_SOIL_LABELS = 23
ROTATE_MICRO_LABELS = 45


# -----------------------------------------------------------------------------
# DIMENSÕES DAS FIGURAS
# -----------------------------------------------------------------------------
# As larguras estão vinculadas a uma largura padrão de figura em duas colunas.
# As alturas foram definidas para preservar o equilíbrio visual e evitar painéis comprimidos.

FIGSIZE_NATIONAL = (DOUBLE_COLUMN_WIDTH, 4.55)
FIGSIZE_MACRO = (DOUBLE_COLUMN_WIDTH, 4.55)
FIGSIZE_MICRO = (DOUBLE_COLUMN_WIDTH * 1.20, 5.60)
FIGSIZE_SOIL = (DOUBLE_COLUMN_WIDTH, 4.85)
FIGSIZE_SOIL_MACRO_BAR = (DOUBLE_COLUMN_WIDTH, 5.00)
FIGSIZE_HEATMAP = (DOUBLE_COLUMN_WIDTH * 1.22, 10.10)
FIGSIZE_MACRO_SOIL_GRID = (DOUBLE_COLUMN_WIDTH * 1.12, 15.0)


# -----------------------------------------------------------------------------
# FIGURA DOS ANOS EXTREMOS NACIONAIS
# -----------------------------------------------------------------------------

REFERENCE_LINE_NATIONAL_MM = 150
N_LOW_YEARS = 2
N_HIGH_YEARS = 2


# -----------------------------------------------------------------------------
# OPÇÕES DOS BOXPLOTS
# -----------------------------------------------------------------------------

SHOW_OUTLIERS = False


# -----------------------------------------------------------------------------
# OPÇÕES DOS HEATMAPS
# -----------------------------------------------------------------------------

HEATMAP_CMAP = "viridis"
SHOW_HEATMAP_VALUES = True


# -----------------------------------------------------------------------------
# LAYOUT MULTIPAINEL
# -----------------------------------------------------------------------------

SHOW_SOIL_NAMES_ONLY_ON_BOTTOM_ROW = True

# -----------------------------------------------------------------------------
# AJUSTES ESPECÍFICOS DE COMPOSIÇÃO
# -----------------------------------------------------------------------------
NATIONAL_TOP_MARGIN_FACTOR = 1.10
MICRO_BOX_WIDTH = 0.78
HEATMAP_TOP_BOTTOM_SPACING = 0.34
ROW_LABEL_X_POSITION = 0.972



# -----------------------------------------------------------------------------
# CONFIGURAÇÕES GLOBAIS DO MATPLOTLIB PARA PUBLICAÇÃO
# -----------------------------------------------------------------------------

mpl.rcParams.update({
    # Font
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_FAMILY, "DejaVu Sans"],
    "font.size": BASE_FONT_SIZE,

    # Axis labels and ticks
    "axes.labelsize": AXIS_TITLE_FONT_SIZE,
    "xtick.labelsize": TICK_FONT_SIZE,
    "ytick.labelsize": TICK_FONT_SIZE,
    "legend.fontsize": LEGEND_FONT_SIZE,
    "axes.linewidth": AXIS_LINE_WIDTH,

    # Export quality
    "figure.dpi": 150,
    "savefig.dpi": DPI,
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "savefig.transparent": False,

    # Font embedding / editability
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    # Sharp lines
    "lines.antialiased": True,
    "patch.antialiased": True,

    # Clean white background
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


In [ ]:
# =============================================================================
# CÉLULA 4 — CORES (MANTER SINCRONIZADAS COM OS MAPAS)
# =============================================================================

# Cores das macrorregiões — mesma paleta usada nos gráficos baseados em mapas.
CORES_MACRO = {
    "Macro_1": "#86B86B",  # verde
    "Macro_2": "#6F96C6",  # azul
    "Macro_3": "#E6B65F",  # ocre/amarelo
    "Macro_4": "#9B8BBE",  # roxo
    "Macro_5": "#C98A63",  # terracota
}

# Grupos do ranking das microrregiões.
CORES_GRUPOS_MICRO = {
    "Grupo 1: maiores déficits": "#d73027",
    "Grupo 2: déficits altos": "#fdae61",
    "Grupo 3: déficits intermediários": "#ffffbf",
    "Grupo 4: menores déficits": "#1a9850",
}

# Cores das classes de solo para gráficos centrados em solos.
CORES_SOLOS = {
    "Neossolo Quartzarênico": "#F3DDB9",
    "Latossolo textura média": "#DEB88E",
    "Latossolo argiloso": "#C99575",
    "Latossolo muito argiloso": "#A88462",
    "Argissolo": "#D9857F",
}

# Cores para os anos extremos nacionais.
CORES_ANOS_EXTREMOS = {
    "baixo": "#1A9850",
    "alto": "#D73027",
    "normal": "#BDBDBD",
}


In [ ]:
# =============================================================================
# CÉLULA 5 — FUNÇÕES DE ENTRADA E SAÍDA
# =============================================================================

def detectar_separador(caminho: Path) -> str:
    """Detect whether the CSV separator is ';' or ','."""
    with open(caminho, "r", encoding="utf-8-sig", errors="ignore") as f:
        primeira = f.readline()
    return ";" if primeira.count(";") > primeira.count(",") else ","


def ler_csv_flexivel(caminho: Path) -> pd.DataFrame:
    """
    Read CSV files exported with either Brazilian or international conventions.
    Colunas do tipo texto que aparentam ser numéricas são convertidas automaticamente.
    """
    caminho = Path(caminho)
    sep = detectar_separador(caminho)

    df = pd.read_csv(
        caminho,
        sep=sep,
        encoding="utf-8-sig",
        low_memory=False,
    )

    for col in df.columns:
        if df[col].dtype != "object":
            continue

        s = df[col].astype(str).str.strip()
        amostra = s.replace({"nan": np.nan, "None": np.nan}).dropna().head(100)

        if amostra.empty:
            continue

        teste = pd.to_numeric(
            amostra.str.replace(",", ".", regex=False),
            errors="coerce",
        )

        if teste.notna().mean() >= 0.80:
            df[col] = pd.to_numeric(
                s.str.replace(",", ".", regex=False),
                errors="coerce",
            )

    return df


def localizar_arquivo(nome: str, obrigatorio=True):
    """
    Search recursively inside data/analysis.
    Se houver múltiplas cópias, a mais recente será selecionada.
    """
    candidatos = list(PASTA_TABELAS_GERAL.rglob(nome))

    if not candidatos:
        if obrigatorio:
            raise FileNotFoundError(
                f"File not found inside {PASTA_TABELAS_GERAL}: {nome}"
            )
        return None

    candidatos = sorted(
        candidatos,
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if len(candidatos) > 1:
        print(f"\nWARNING — multiple copies of {nome}:")
        for p in candidatos:
            print(" -", p)
        print("Using the newest copy:", candidatos[0])

    return candidatos[0]


def salvar_figura(fig, nome_base: str):
    """
    Save each figure in publication-quality formats.

    Saídas:
    - PNG: raster em 600 dpi
    - TIFF: raster em 600 dpi, com compressão LZW quando suportado
    - PDF: vetorial
    - SVG: vetorial com texto editável

    Os formatos vetoriais devem ser priorizados sempre que o periódico permitir.
    """
    # Usa uma caixa delimitadora ajustada, preservando uma pequena margem de segurança.
    common_kwargs = {
        "bbox_inches": "tight",
        "pad_inches": 0.04,
        "facecolor": "white",
        "edgecolor": "white",
    }

    if SALVAR_PNG:
        caminho = PASTA_FIGURAS / f"{nome_base}.png"
        fig.savefig(
            caminho,
            dpi=DPI,
            **common_kwargs,
        )
        print("Figura salva:", caminho)

    if SALVAR_TIFF:
        caminho = PASTA_FIGURAS / f"{nome_base}.tif"
        try:
            fig.savefig(
                caminho,
                dpi=DPI,
                pil_kwargs={"compression": "tiff_lzw"},
                **common_kwargs,
            )
        except TypeError:
            # Alternativa para versões mais antigas do matplotlib/Pillow.
            fig.savefig(
                caminho,
                dpi=DPI,
                **common_kwargs,
            )
        print("Figura salva:", caminho)

    if SALVAR_PDF:
        caminho = PASTA_FIGURAS / f"{nome_base}.pdf"
        fig.savefig(
            caminho,
            **common_kwargs,
        )
        print("Figura salva:", caminho)

    if SALVAR_SVG:
        caminho = PASTA_FIGURAS / f"{nome_base}.svg"
        fig.savefig(
            caminho,
            **common_kwargs,
        )
        print("Figura salva:", caminho)

    if MOSTRAR_FIGURAS:
        plt.show()
    else:
        plt.close(fig)


def sufixo_titulos_eixos(com_titulos_eixos: bool) -> str:
    """Retorna o sufixo usado nos nomes dos arquivos exportados."""
    return "with_axis_titles" if com_titulos_eixos else "no_axis_titles"


In [ ]:
# =============================================================================
# CÉLULA 6 — FUNÇÕES AUXILIARES GERAIS
# =============================================================================

def normalizar_texto(s: str) -> str:
    s = str(s).strip().lower()
    s = (
        s.replace("á", "a").replace("à", "a").replace("ã", "a").replace("â", "a")
         .replace("é", "e").replace("ê", "e")
         .replace("í", "i")
         .replace("ó", "o").replace("ô", "o").replace("õ", "o")
         .replace("ú", "u")
         .replace("ç", "c")
    )
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def mapa_colunas_normalizadas(df: pd.DataFrame) -> dict:
    return {normalizar_texto(c): c for c in df.columns}


def localizar_coluna(df: pd.DataFrame, aliases, obrigatoria=True):
    """Localiza uma coluna a partir de uma lista de possíveis aliases."""
    mapa = mapa_colunas_normalizadas(df)
    for alias in aliases:
        key = normalizar_texto(alias)
        if key in mapa:
            return mapa[key]
    if obrigatoria:
        raise KeyError(
            "None of the following columns was found:\n"
            f"{aliases}\nAvailable columns:\n{list(df.columns)}"
        )
    return None


def para_numerico(s: pd.Series) -> pd.Series:
    """Converte uma série para numérico de forma segura."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(
        s.astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )


def media_ponderada(df, valor, peso):
    """Calcula média ponderada ignorando valores ausentes."""
    d = df[[valor, peso]].copy()
    d[valor] = para_numerico(d[valor])
    d[peso] = para_numerico(d[peso])
    d = d.dropna()
    d = d.loc[d[peso] > 0]

    if d.empty:
        return np.nan

    return np.average(d[valor], weights=d[peso])


def aplicar_estilo_eixos(ax, eixo_grade="y"):
    """Aplica um estilo limpo e consistente de publicação aos eixos."""
    ax.grid(
        True,
        axis=eixo_grade,
        linestyle="-",
        linewidth=GRID_LINE_WIDTH,
        alpha=GRID_ALPHA,
    )
    ax.set_axisbelow(True)
    ax.tick_params(width=AXIS_LINE_WIDTH)
    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINE_WIDTH)


def formatar_nome_macro(macro: str) -> str:
    """Converte Macro_1 em Macro 1."""
    return str(macro).replace("_", " ")


def formatar_legenda_macro(macro: str) -> str:
    """Retorna o rótulo da legenda das macrorregiões sem underscore."""
    return formatar_nome_macro(macro)


def definir_titulos_eixos(ax, com_titulos_eixos: bool, xlabel="", ylabel=""):
    """Ativa ou desativa os títulos dos eixos x/y."""
    ax.set_xlabel(xlabel if com_titulos_eixos else "")
    ax.set_ylabel(ylabel if com_titulos_eixos else "")


def cor_micro_por_ranking(row):
    """
    Dá preferência à cor HEX explícita da tabela de ranking.
    Se a HEX estiver ausente, usa a paleta do grupo de ranking.
    """
    hex_col = str(row.get("hex", "")).strip()
    if re.fullmatch(r"#[0-9a-fA-F]{6}", hex_col):
        return hex_col

    grupo = str(row.get("grupo", "")).strip()
    for nome_grupo, cor in CORES_GRUPOS_MICRO.items():
        if grupo.lower() == nome_grupo.lower():
            return cor

    return "#BDBDBD"


def desenhar_boxplot_colorido(
    ax,
    df,
    categoria_col,
    valor_col,
    ordem,
    mapa_cores,
    rotacao_xticks=0,
    largura=0.55,
    rotulos_display=None,
):
    """
    Desenha um boxplot colorido e limpo usando apenas matplotlib.
    Os pontos de outliers ficam desativados, conforme solicitado.
    """
    dados = []
    categorias_validas = []

    for cat in ordem:
        sub = df.loc[df[categoria_col].astype(str) == str(cat), valor_col]
        sub = para_numerico(sub).dropna()
        if len(sub) == 0:
            continue
        dados.append(sub.values)
        categorias_validas.append(cat)

    posicoes = np.arange(1, len(categorias_validas) + 1)

    bp = ax.boxplot(
        dados,
        positions=posicoes,
        widths=largura,
        patch_artist=True,
        showfliers=SHOW_OUTLIERS,
        boxprops=dict(linewidth=BOX_LINE_WIDTH),
        medianprops=dict(linewidth=MEDIAN_LINE_WIDTH, color="black"),
        whiskerprops=dict(linewidth=WHISKER_LINE_WIDTH),
        capprops=dict(linewidth=CAP_LINE_WIDTH),
    )

    for patch, cat in zip(bp["boxes"], categorias_validas):
        patch.set_facecolor(mapa_cores.get(cat, "#BDBDBD"))
        patch.set_alpha(BOX_ALPHA)
        patch.set_edgecolor("black")

    ax.set_xticks(posicoes)

    if rotulos_display is None:
        labels = [str(x) for x in categorias_validas]
    else:
        labels = [rotulos_display.get(x, str(x)) for x in categorias_validas]

    ax.set_xticklabels(
        labels,
        rotation=rotacao_xticks,
        ha="right" if rotacao_xticks else "center",
        rotation_mode="anchor",
    )
    aplicar_estilo_eixos(ax, eixo_grade="y")

    return categorias_validas


In [ ]:
# =============================================================================
# CÉLULA 7 — CARREGAR TABELAS DE ENTRADA
# =============================================================================

# Localiza os arquivos necessários.
CAMINHO_DEF_MACRO = localizar_arquivo(ARQ_DEF_MACRO)
CAMINHO_DEF_MICRO = localizar_arquivo(ARQ_DEF_MICRO)
CAMINHO_RANKING_MICRO = localizar_arquivo(ARQ_RANKING_MICRO)
CAMINHO_DEF_SOLOS = localizar_arquivo(ARQ_DEF_SOLOS)
CAMINHO_DEF_SOLO_MACRO = localizar_arquivo(ARQ_DEF_SOLO_MACRO)
CAMINHO_SOLO_ANUAL_DEF_PROD = localizar_arquivo(ARQ_SOLO_ANUAL_DEF_PROD)
CAMINHO_SOLO_MACRO_ANUAL_DEF_PROD = localizar_arquivo(ARQ_SOLO_MACRO_ANUAL_DEF_PROD)

# Lê as tabelas.
df_macro_raw = ler_csv_flexivel(CAMINHO_DEF_MACRO)
df_micro_raw = ler_csv_flexivel(CAMINHO_DEF_MICRO)
df_ranking_raw = ler_csv_flexivel(CAMINHO_RANKING_MICRO)
df_solos_raw = ler_csv_flexivel(CAMINHO_DEF_SOLOS)
df_solo_macro_raw = ler_csv_flexivel(CAMINHO_DEF_SOLO_MACRO)
df_solo_anual_prod_raw = ler_csv_flexivel(CAMINHO_SOLO_ANUAL_DEF_PROD)
df_solo_macro_anual_prod_raw = ler_csv_flexivel(CAMINHO_SOLO_MACRO_ANUAL_DEF_PROD)

print("\nArquivos de entrada utilizados:")
for caminho in [
    CAMINHO_DEF_MACRO,
    CAMINHO_DEF_MICRO,
    CAMINHO_RANKING_MICRO,
    CAMINHO_DEF_SOLOS,
    CAMINHO_DEF_SOLO_MACRO,
    CAMINHO_SOLO_ANUAL_DEF_PROD,
    CAMINHO_SOLO_MACRO_ANUAL_DEF_PROD,
]:
    print(" -", caminho)


In [ ]:
# =============================================================================
# CÉLULA 8 — PADRONIZAR TABELAS
# =============================================================================

def padronizar_def_macro(df):
    return pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "macro": df[localizar_coluna(df, ["macro", "macroregiao", "macrorregiao"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "n_pixels": para_numerico(df[localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False)]) if localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False) else 1.0,
    })


def padronizar_def_micro(df):
    return pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "codigo_micro": para_numerico(df[localizar_coluna(df, ["codigo_micro", "micro"])]).astype("Int64"),
        "macro": df[localizar_coluna(df, ["macro", "macroregiao", "macrorregiao"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "n_pixels": para_numerico(df[localizar_coluna(df, ["n_pixels", "pixel_count"])]),
    })


def padronizar_ranking_micro(df):
    col_micro = localizar_coluna(df, ["codigo_micro", "micro"])
    col_rank = localizar_coluna(df, ["ranking", "rank"])
    col_group = localizar_coluna(df, ["grupo", "group"], obrigatoria=False)
    col_hex = localizar_coluna(df, ["hex", "color_hex", "cor"], obrigatoria=False)

    out = pd.DataFrame({
        "codigo_micro": para_numerico(df[col_micro]).astype("Int64"),
        "ranking": para_numerico(df[col_rank]),
    })

    out["grupo"] = df[col_group].astype(str).str.strip() if col_group else ""
    out["hex"] = df[col_hex].astype(str).str.strip() if col_hex else ""

    out = out.dropna(subset=["codigo_micro"]).drop_duplicates("codigo_micro")
    out = out.sort_values("ranking").reset_index(drop=True)
    return out


def padronizar_def_solos(df):
    return pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "classe_solo": df[localizar_coluna(df, ["classe_solo", "solo", "soil_class"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "n_pixels": para_numerico(df[localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False)]) if localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False) else 1.0,
    })


def padronizar_def_solo_macro(df):
    return pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "classe_solo": df[localizar_coluna(df, ["classe_solo", "solo", "soil_class"])].astype(str).str.strip(),
        "macro": df[localizar_coluna(df, ["macro", "macroregiao", "macrorregiao"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "n_pixels": para_numerico(df[localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False)]) if localizar_coluna(df, ["n_pixels", "pixel_count"], obrigatoria=False) else 1.0,
    })


def padronizar_solo_anual_def_prod(df):
    out = pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "classe_solo": df[localizar_coluna(df, ["classe_solo", "solo", "soil_class"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "prod_sc_ha_mean": para_numerico(df[localizar_coluna(df, ["prod_sc_ha_mean", "yield_sc_ha", "produtividade_sc_ha"])]),
        "prod_kg_ha_mean": para_numerico(df[localizar_coluna(df, ["prod_kg_ha_mean", "yield_kg_ha", "produtividade_kg_ha"], obrigatoria=False)]) if localizar_coluna(df, ["prod_kg_ha_mean", "yield_kg_ha", "produtividade_kg_ha"], obrigatoria=False) else np.nan,
        "area_soja_solo_ha": para_numerico(df[localizar_coluna(df, ["area_soja_solo_ha", "area_ha"], obrigatoria=False)]) if localizar_coluna(df, ["area_soja_solo_ha", "area_ha"], obrigatoria=False) else np.nan,
    })

    # Conversão da produtividade para Mg ha^-1, unidade usada no artigo.
    if out["prod_kg_ha_mean"].notna().any():
        out["prod_Mg_ha"] = out["prod_kg_ha_mean"] / 1000.0
    else:
        out["prod_Mg_ha"] = out["prod_sc_ha_mean"] * 0.06

    return out


def padronizar_solo_macro_anual_def_prod(df):
    out = pd.DataFrame({
        "ano": para_numerico(df[localizar_coluna(df, ["ano", "year"])]),
        "macro": df[localizar_coluna(df, ["macro", "macroregiao", "macrorregiao"])].astype(str).str.strip(),
        "classe_solo": df[localizar_coluna(df, ["classe_solo", "solo", "soil_class"])].astype(str).str.strip(),
        "def_mean": para_numerico(df[localizar_coluna(df, ["def_mean", "mean_def", "mean_def_acum"])]),
        "prod_sc_ha_mean": para_numerico(df[localizar_coluna(df, ["prod_sc_ha_mean", "yield_sc_ha", "produtividade_sc_ha"])]),
        "prod_kg_ha_mean": para_numerico(df[localizar_coluna(df, ["prod_kg_ha_mean", "yield_kg_ha", "produtividade_kg_ha"], obrigatoria=False)]) if localizar_coluna(df, ["prod_kg_ha_mean", "yield_kg_ha", "produtividade_kg_ha"], obrigatoria=False) else np.nan,
        "area_soja_solo_ha": para_numerico(df[localizar_coluna(df, ["area_soja_solo_ha", "area_ha"], obrigatoria=False)]) if localizar_coluna(df, ["area_soja_solo_ha", "area_ha"], obrigatoria=False) else np.nan,
    })

    # Conversão da produtividade para Mg ha^-1, unidade usada no artigo.
    if out["prod_kg_ha_mean"].notna().any():
        out["prod_Mg_ha"] = out["prod_kg_ha_mean"] / 1000.0
    else:
        out["prod_Mg_ha"] = out["prod_sc_ha_mean"] * 0.06

    return out


df_macro = padronizar_def_macro(df_macro_raw)
df_micro = padronizar_def_micro(df_micro_raw)
df_ranking = padronizar_ranking_micro(df_ranking_raw)
df_solos = padronizar_def_solos(df_solos_raw)
df_solo_macro = padronizar_def_solo_macro(df_solo_macro_raw)
df_solo_anual_prod = padronizar_solo_anual_def_prod(df_solo_anual_prod_raw)
df_solo_macro_anual_prod = padronizar_solo_macro_anual_def_prod(df_solo_macro_anual_prod_raw)

# Mantém apenas as categorias e anos esperados.
df_macro = df_macro.loc[
    df_macro["macro"].isin(MACROS_ORDEM) & df_macro["ano"].isin(ANOS_ESPERADOS)
].copy()

df_micro = df_micro.loc[df_micro["ano"].isin(ANOS_ESPERADOS)].copy()

df_solos = df_solos.loc[
    df_solos["classe_solo"].isin(ORDEM_SOLOS) & df_solos["ano"].isin(ANOS_ESPERADOS)
].copy()

df_solo_macro = df_solo_macro.loc[
    df_solo_macro["classe_solo"].isin(ORDEM_SOLOS)
    & df_solo_macro["macro"].isin(MACROS_ORDEM)
    & df_solo_macro["ano"].isin(ANOS_ESPERADOS)
].copy()

df_solo_anual_prod = df_solo_anual_prod.loc[
    df_solo_anual_prod["classe_solo"].isin(ORDEM_SOLOS)
    & df_solo_anual_prod["ano"].isin(ANOS_ESPERADOS)
].copy()

df_solo_macro_anual_prod = df_solo_macro_anual_prod.loc[
    df_solo_macro_anual_prod["classe_solo"].isin(ORDEM_SOLOS)
    & df_solo_macro_anual_prod["macro"].isin(MACROS_ORDEM)
    & df_solo_macro_anual_prod["ano"].isin(ANOS_ESPERADOS)
].copy()

print("\nTabelas padronizadas:")
print("df_macro:", df_macro.shape)
print("df_micro:", df_micro.shape)
print("df_ranking:", df_ranking.shape)
print("df_solos:", df_solos.shape)
print("df_solo_macro:", df_solo_macro.shape)
print("df_solo_anual_prod:", df_solo_anual_prod.shape)
print("df_solo_macro_anual_prod:", df_solo_macro_anual_prod.shape)


In [ ]:
# =============================================================================
# CÉLULA 9 — PREPARAR TABELAS AUXILIARES
# =============================================================================

# Ordem e cores das microrregiões com base na tabela de ranking.
df_ranking["cor"] = df_ranking.apply(cor_micro_por_ranking, axis=1)

ORDEM_MICRO = (
    df_ranking["codigo_micro"]
    .dropna()
    .astype(int)
    .tolist()
)

CORES_MICRO_ORDENADAS = {
    int(row["codigo_micro"]): row["cor"]
    for _, row in df_ranking.dropna(subset=["codigo_micro"]).iterrows()
}

# Média histórica de DEF por solo × macrorregião.
linhas_solo_macro_historico = []
for (solo, macro), g in df_solo_macro.groupby(["classe_solo", "macro"], observed=True):
    linhas_solo_macro_historico.append({
        "classe_solo": solo,
        "macro": macro,
        "def_mean_historico": media_ponderada(g, "def_mean", "n_pixels"),
    })

df_solo_macro_historico = pd.DataFrame(linhas_solo_macro_historico)

# Médias anuais nacionais.
linhas_nacionais = []
for ano, g in df_micro.groupby("ano", observed=True):
    linhas_nacionais.append({
        "ano": int(ano),
        "def_mean": media_ponderada(g, "def_mean", "n_pixels"),
    })

df_nacional = pd.DataFrame(linhas_nacionais).sort_values("ano").reset_index(drop=True)

print("\nTabelas auxiliares preparadas:")
print("ORDEM_MICRO length:", len(ORDEM_MICRO))
print("df_solo_macro_historico:", df_solo_macro_historico.shape)
print("df_nacional:", df_nacional.shape)


In [ ]:
# =============================================================================
# CÉLULA 10 — FIGURA 1
# Anos extremos nacionais de DEF_ACUM
# =============================================================================

def figura_01_anos_extremos_nacionais(com_titulos_eixos: bool):
    """
    Gráfico de barras das médias anuais nacionais ponderadas de DEF_ACUM.
    Os anos mais baixos e mais altos são destacados.
    """
    tab = df_nacional.dropna().copy()

    anos_baixos = set(tab.nsmallest(N_LOW_YEARS, "def_mean")["ano"].astype(int))
    anos_altos = set(tab.nlargest(N_HIGH_YEARS, "def_mean")["ano"].astype(int))

    cores = []
    for ano in tab["ano"].astype(int):
        if ano in anos_baixos:
            cores.append(CORES_ANOS_EXTREMOS["baixo"])
        elif ano in anos_altos:
            cores.append(CORES_ANOS_EXTREMOS["alto"])
        else:
            cores.append(CORES_ANOS_EXTREMOS["normal"])

    fig, ax = plt.subplots(figsize=FIGSIZE_NATIONAL)

    barras = ax.bar(
        tab["ano"].astype(str),
        tab["def_mean"],
        color=cores,
        edgecolor="0.25",
        linewidth=BAR_EDGE_WIDTH,
        alpha=BAR_ALPHA,
    )

    ax.axhline(
        REFERENCE_LINE_NATIONAL_MM,
        color="0.35",
        linestyle="--",
        linewidth=1.1,
    )

    y_offset = tab["def_mean"].max() * 0.015 if not tab.empty else 1.0

    for barra, ano, valor in zip(barras, tab["ano"].astype(int), tab["def_mean"]):
        if ano in anos_baixos or ano in anos_altos:
            ax.text(
                barra.get_x() + barra.get_width() / 2,
                valor + y_offset,
                f"{valor:.0f}",
                ha="center",
                va="bottom",
                fontsize=ANNOTATION_FONT_SIZE,
                fontweight="bold",
            )

    definir_titulos_eixos(
        ax,
        com_titulos_eixos,
        xlabel="IBGE year",
        ylabel="Pixel-weighted mean accumulated water deficit (mm)",
    )
    aplicar_estilo_eixos(ax, eixo_grade="y")
    ax.margins(x=0.04)

    # Garante espaço superior para as anotações dos anos extremos.
    if not tab.empty:
        topo_atual = max(tab["def_mean"].max(), REFERENCE_LINE_NATIONAL_MM)
        ax.set_ylim(0, topo_atual * NATIONAL_TOP_MARGIN_FACTOR)

    salvar_figura(
        fig,
        f"Fig24_01_national_extreme_years_{sufixo_titulos_eixos(com_titulos_eixos)}"
    )


In [ ]:
# =============================================================================
# CÉLULA 11 — FIGURA 2
# DEF_ACUM por macrorregião
# =============================================================================

def figura_02_boxplot_macro(com_titulos_eixos: bool):
    """Distribuição histórica de DEF_ACUM por macrorregião da soja."""
    fig, ax = plt.subplots(figsize=FIGSIZE_MACRO)

    desenhar_boxplot_colorido(
        ax=ax,
        df=df_macro,
        categoria_col="macro",
        valor_col="def_mean",
        ordem=MACROS_ORDEM,
        mapa_cores=CORES_MACRO,
        rotacao_xticks=0,
        largura=0.52,
    )

    # Substitui os rótulos do eixo x por 'Macro 1', ..., 'Macro 5'
    ax.set_xticklabels([formatar_nome_macro(m) for m in MACROS_ORDEM])

    definir_titulos_eixos(
        ax,
        com_titulos_eixos,
        xlabel="Soybean macroregion",
        ylabel="Mean accumulated water deficit (mm)",
    )

    salvar_figura(
        fig,
        f"Fig24_02_def_by_macroregion_{sufixo_titulos_eixos(com_titulos_eixos)}"
    )


In [ ]:
# =============================================================================
# CÉLULA 12 — FIGURA 3
# DEF_ACUM por microrregião
# =============================================================================

def figura_03_boxplot_micro(com_titulos_eixos: bool):
    """
    Distribuição histórica de DEF_ACUM por região edafoclimática,
    ordenada pelo ranking histórico.
    """
    df_plot = df_micro.copy()
    df_plot["codigo_micro"] = df_plot["codigo_micro"].astype("Int64")

    fig, ax = plt.subplots(figsize=FIGSIZE_MICRO)

    desenhar_boxplot_colorido(
        ax=ax,
        df=df_plot,
        categoria_col="codigo_micro",
        valor_col="def_mean",
        ordem=ORDEM_MICRO,
        mapa_cores=CORES_MICRO_ORDENADAS,
        rotacao_xticks=ROTATE_MICRO_LABELS,
        largura=MICRO_BOX_WIDTH,
    )

    definir_titulos_eixos(
        ax,
        com_titulos_eixos,
        xlabel="Edaphoclimatic region code",
        ylabel="Mean accumulated water deficit (mm)",
    )

    salvar_figura(
        fig,
        f"Fig24_03_def_by_microregion_{sufixo_titulos_eixos(com_titulos_eixos)}"
    )


In [ ]:
# =============================================================================
# CÉLULA 13 — FIGURA 4
# DEF_ACUM por classe de solo
# Os rótulos visíveis seguem a terminologia WRB usada no manuscrito.
# =============================================================================

def figura_04_boxplot_solos(com_titulos_eixos: bool):
    """Distribuição histórica de DEF_ACUM por classe de solo."""
    fig, ax = plt.subplots(figsize=FIGSIZE_SOIL)

    desenhar_boxplot_colorido(
        ax=ax,
        df=df_solos,
        categoria_col="classe_solo",
        valor_col="def_mean",
        ordem=ORDEM_SOLOS,
        mapa_cores=CORES_SOLOS,
        rotacao_xticks=ROTATE_SOIL_LABELS,
        largura=0.52,
        rotulos_display=ROTULOS_SOLOS_INGLES,
    )

    definir_titulos_eixos(
        ax,
        com_titulos_eixos,
        xlabel="Soil class",
        ylabel="Mean accumulated water deficit (mm)",
    )

    salvar_figura(
        fig,
        f"Fig24_04_def_by_soil_class_{sufixo_titulos_eixos(com_titulos_eixos)}"
    )


In [ ]:
# =============================================================================
# CÉLULA 14 — FIGURA 5
# DEF_ACUM histórico por classe de solo e macrorregião
# =============================================================================

def figura_05_barras_solo_macro(com_titulos_eixos: bool):
    """
    Barras agrupadas da média histórica de DEF_ACUM por classe de solo e macrorregião.

    IMPORTANTE:
    As cores das macrorregiões são exatamente as mesmas da paleta usada no mapa das macrorregiões.
    """
    tab = (
        df_solo_macro_historico
        .pivot(index="classe_solo", columns="macro", values="def_mean_historico")
        .reindex(index=ORDEM_SOLOS, columns=MACROS_ORDEM)
    )

    fig, ax = plt.subplots(figsize=FIGSIZE_SOIL_MACRO_BAR)

    x = np.arange(len(ORDEM_SOLOS))
    largura = 0.14

    for i, macro in enumerate(MACROS_ORDEM):
        desloc = (i - (len(MACROS_ORDEM) - 1) / 2) * largura
        valores = tab[macro].to_numpy(dtype=float)
        ax.bar(
            x + desloc,
            valores,
            width=largura,
            color=CORES_MACRO.get(macro, "#BDBDBD"),
            edgecolor="0.30",
            linewidth=BAR_EDGE_WIDTH,
            alpha=BAR_ALPHA,
            label=formatar_legenda_macro(macro),
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [ROTULOS_SOLOS_INGLES.get(solo, solo) for solo in ORDEM_SOLOS],
        rotation=ROTATE_SOIL_LABELS,
        ha="right",
        rotation_mode="anchor",
    )

    definir_titulos_eixos(
        ax,
        com_titulos_eixos,
        xlabel="Soil class",
        ylabel="Historical mean accumulated water deficit (mm)",
    )
    aplicar_estilo_eixos(ax, eixo_grade="y")

    # Reserva uma faixa lateral para acomodar a legenda externa.
    fig.subplots_adjust(right=0.79)

    # Posiciona a legenda fora da área do gráfico, à direita.
    # O anchor é definido em coordenadas da figura para manter a legenda
    # paralela à borda direita e sem sobrepor as barras.
    leg = ax.legend(
        title="Soybean macroregion",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.01, 0.98),
        borderaxespad=0.0,
    )
    if leg is not None:
        plt.setp(leg.get_title(), fontsize=LEGEND_TITLE_FONT_SIZE)
        plt.setp(leg.get_texts(), fontsize=LEGEND_FONT_SIZE)

    # Salva manualmente esta figura para preservar espaço suficiente
    # para a legenda externa à direita.
    nome_base = f"Fig24_05_def_by_soil_and_macroregion_{sufixo_titulos_eixos(com_titulos_eixos)}"

    common_kwargs = {
        "bbox_inches": "tight",
        "pad_inches": 0.06,
        "facecolor": "white",
        "edgecolor": "white",
    }

    if SALVAR_PNG:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.png", dpi=DPI, **common_kwargs)
    if SALVAR_TIFF:
        try:
            fig.savefig(
                PASTA_FIGURAS / f"{nome_base}.tif",
                dpi=DPI,
                pil_kwargs={"compression": "tiff_lzw"},
                **common_kwargs,
            )
        except TypeError:
            fig.savefig(PASTA_FIGURAS / f"{nome_base}.tif", dpi=DPI, **common_kwargs)
    if SALVAR_PDF:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.pdf", **common_kwargs)
    if SALVAR_SVG:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.svg", **common_kwargs)

    print("Figura salva:", PASTA_FIGURAS / nome_base)

    if MOSTRAR_FIGURAS:
        plt.show()
    else:
        plt.close(fig)


In [ ]:
# =============================================================================
# CÉLULA 15 — FIGURA 6
# Heatmaps: DEF_ACUM anual e produtividade anual da soja entre classes de solo
# =============================================================================

def cor_texto_heatmap(valor, vmin, vmax):
    """
    Escolhe a cor da anotação com base na intensidade do valor para garantir contraste básico.
    """
    if not np.isfinite(valor) or not np.isfinite(vmin) or not np.isfinite(vmax):
        return "black"
    if vmax == vmin:
        return "black"
    frac = (valor - vmin) / (vmax - vmin)
    return "white" if frac < 0.35 else "black"


def desenhar_heatmap(
    ax,
    matriz,
    formato,
    rotulo_cbar,
    com_titulos_eixos,
    mostrar_xlabel,
    mostrar_ylabel,
):
    """
    Desenha um painel de heatmap com anotações numéricas opcionais.
    """
    arr = matriz.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(arr)

    im = ax.imshow(
        masked,
        aspect="auto",
        interpolation="nearest",
        cmap=HEATMAP_CMAP,
    )

    ax.set_xticks(np.arange(len(matriz.columns)))
    ax.set_xticklabels(
        [str(int(x)) for x in matriz.columns],
        fontsize=HEATMAP_AXIS_TICK_FONT_SIZE,
    )

    ax.set_yticks(np.arange(len(matriz.index)))
    ax.set_yticklabels(
        [ROTULOS_SOLOS_INGLES.get(solo, solo) for solo in matriz.index],
        fontsize=HEATMAP_AXIS_TICK_FONT_SIZE,
        rotation=20,
        ha="right",
        va="center",
        rotation_mode="anchor",
    )

    if com_titulos_eixos and mostrar_xlabel:
        ax.set_xlabel(
            "Year",
            fontsize=HEATMAP_AXIS_LABEL_FONT_SIZE,
        )
    else:
        ax.set_xlabel("")

    if com_titulos_eixos and mostrar_ylabel:
        ax.set_ylabel(
            "Soil class",
             fontsize=HEATMAP_AXIS_LABEL_FONT_SIZE,
        )
    else:
        ax.set_ylabel("")

    cbar = ax.figure.colorbar(im, ax=ax, pad=0.018, fraction=0.035)
    cbar.set_label(rotulo_cbar, fontsize=HEATMAP_CBAR_LABEL_FONT_SIZE)
    cbar.ax.tick_params(labelsize=HEATMAP_CBAR_TICK_FONT_SIZE)

    if SHOW_HEATMAP_VALUES:
        finitos = arr[np.isfinite(arr)]
        if finitos.size:
            vmin, vmax = finitos.min(), finitos.max()
            for i in range(arr.shape[0]):
                for j in range(arr.shape[1]):
                    valor = arr[i, j]
                    if np.isfinite(valor):
                        ax.text(
                            j,
                            i,
                            format(valor, formato),
                            ha="center",
                            va="center",
                            fontsize=HEATMAP_CELL_FONT_SIZE,
                            color=cor_texto_heatmap(valor, vmin, vmax),
                        )

    ax.tick_params(width=AXIS_LINE_WIDTH)
    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINE_WIDTH)


def figura_06_heatmaps_solos(com_titulos_eixos: bool):
    """
    Dois heatmaps empilhados:
    A) déficit hídrico acumulado anual por classe de solo
    B) produtividade anual da soja sequeiro por classe de solo
    """
    matriz_def = (
        df_solo_anual_prod
        .pivot(index="classe_solo", columns="ano", values="def_mean")
        .reindex(index=ORDEM_SOLOS, columns=ANOS_ESPERADOS)
    )

    matriz_prod = (
        df_solo_anual_prod
        .pivot(index="classe_solo", columns="ano", values="prod_Mg_ha")
        .reindex(index=ORDEM_SOLOS, columns=ANOS_ESPERADOS)
    )

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=FIGSIZE_HEATMAP,
        gridspec_kw={"hspace": HEATMAP_TOP_BOTTOM_SPACING},
    )

    desenhar_heatmap(
        axes[0],
        matriz_def,
        ".0f",
        "Accumulated water deficit (mm)",
        com_titulos_eixos=com_titulos_eixos,
        mostrar_xlabel=False,
        mostrar_ylabel=True,
    )
    axes[0].tick_params(axis="x", labelbottom=False)

    desenhar_heatmap(
        axes[1],
        matriz_prod,
        ".2f",
        "Soybean yield (Mg ha$^{-1}$)",
        com_titulos_eixos=com_titulos_eixos,
        mostrar_xlabel=True,
        mostrar_ylabel=True,
    )

    fig.subplots_adjust(
        left=0.20,
        right=0.93,
        bottom=0.08,
        top=0.98,
        hspace=0.30,
    )

    salvar_figura(
        fig,
        f"Fig24_06_annual_def_and_yield_by_soil_{sufixo_titulos_eixos(com_titulos_eixos)}"
    )


In [ ]:
# =============================================================================
# CÉLULA 16 — FIGURA 7
# Boxplots por classe de solo dentro de cada macrorregião
# =============================================================================

def figura_07_paineis_solo_dentro_macro(com_titulos_eixos: bool):
    """
    Regras de layout solicitadas pelo usuário:
    - 5 linhas, uma para cada macrorregião.
    - Coluna da esquerda: DEF_ACUM.
    - Coluna da direita: produtividade da soja.
    - Sem pontos de dados internos.
    - Nomes dos solos mostrados apenas uma vez na parte inferior.
    - Títulos do eixo y mostrados apenas uma vez para cada métrica (em nível de figura).
    - Nome da macrorregião mostrado verticalmente uma vez à direita de cada linha.
    - Sem títulos de painéis.
    """
    fig, axes = plt.subplots(
        nrows=len(MACROS_ORDEM),
        ncols=2,
        figsize=FIGSIZE_MACRO_SOIL_GRID,
        squeeze=False,
    )

    for i, macro in enumerate(MACROS_ORDEM):
        sub = df_solo_macro_anual_prod.loc[
            df_solo_macro_anual_prod["macro"].astype(str) == macro
        ].copy()

        ax_def = axes[i, 0]
        ax_prod = axes[i, 1]

        desenhar_boxplot_colorido(
            ax=ax_def,
            df=sub,
            categoria_col="classe_solo",
            valor_col="def_mean",
            ordem=ORDEM_SOLOS,
            mapa_cores=CORES_SOLOS,
            rotacao_xticks=ROTATE_SOIL_LABELS,
            largura=0.58,
            rotulos_display=ROTULOS_SOLOS_INGLES,
        )

        desenhar_boxplot_colorido(
            ax=ax_prod,
            df=sub,
            categoria_col="classe_solo",
            valor_col="prod_Mg_ha",
            ordem=ORDEM_SOLOS,
            mapa_cores=CORES_SOLOS,
            rotacao_xticks=ROTATE_SOIL_LABELS,
            largura=0.58,
            rotulos_display=ROTULOS_SOLOS_INGLES,
        )

        # Sem títulos individuais de eixos x/y dentro de cada painel.
        ax_def.set_xlabel("")
        ax_prod.set_xlabel("")
        ax_def.set_ylabel("")
        ax_prod.set_ylabel("")

        # Sem títulos nos painéis.
        ax_def.set_title("")
        ax_prod.set_title("")

        # Mostra os nomes dos solos apenas na última linha para manter o layout limpo.
        if SHOW_SOIL_NAMES_ONLY_ON_BOTTOM_ROW and i < len(MACROS_ORDEM) - 1:
            ax_def.tick_params(axis="x", labelbottom=False)
            ax_prod.tick_params(axis="x", labelbottom=False)
        else:
            ax_def.tick_params(axis="x", rotation=ROTATE_SOIL_LABELS)
            ax_prod.tick_params(axis="x", rotation=ROTATE_SOIL_LABELS)


    # Adiciona títulos de eixos em nível de figura apenas na versão com títulos dos eixos.
    if com_titulos_eixos:
        fig.text(
            0.024,
            0.51,
            "Accumulated water deficit (mm)",
            rotation=90,
            va="center",
            ha="center",
            fontsize=AXIS_TITLE_FONT_SIZE,
        )
        fig.text(
            0.485,
            0.51,
            "Soybean yield (Mg ha$^{-1}$)",
            rotation=90,
            va="center",
            ha="center",
            fontsize=AXIS_TITLE_FONT_SIZE,
        )
        fig.supxlabel(
            "Soil class",
            fontsize=AXIS_TITLE_FONT_SIZE,
            y=0.025,
        )

    fig.subplots_adjust(
        left=0.085,
        right=0.945,
        bottom=0.115,
        top=0.985,
        wspace=0.28,
        hspace=0.16,
    )

    # Posiciona os rótulos das macrorregiões somente após o layout final,
    # para alinhar exatamente cada rótulo com o centro da sua fileira.
    for i, macro in enumerate(MACROS_ORDEM):
        pos = axes[i, 1].get_position()
        y_centro = (pos.y0 + pos.y1) / 2
        fig.text(
            ROW_LABEL_X_POSITION,
            y_centro,
            formatar_nome_macro(macro),
            rotation=270,
            ha="center",
            va="center",
            fontsize=ROW_LABEL_FONT_SIZE,
            fontweight="bold",
        )

    # Não usar tight_layout automático aqui porque os rótulos em nível de figura
    # e os rótulos verticais das macrorregiões são posicionados manualmente.
    nome_base = (
        "Fig24_07_def_and_yield_by_soil_within_macroregions_"
        f"{sufixo_titulos_eixos(com_titulos_eixos)}"
    )

    common_kwargs = {
        "bbox_inches": "tight",
        "pad_inches": 0.04,
        "facecolor": "white",
        "edgecolor": "white",
    }

    if SALVAR_PNG:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.png", dpi=DPI, **common_kwargs)
    if SALVAR_TIFF:
        try:
            fig.savefig(
                PASTA_FIGURAS / f"{nome_base}.tif",
                dpi=DPI,
                pil_kwargs={"compression": "tiff_lzw"},
                **common_kwargs,
            )
        except TypeError:
            fig.savefig(PASTA_FIGURAS / f"{nome_base}.tif", dpi=DPI, **common_kwargs)
    if SALVAR_PDF:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.pdf", **common_kwargs)
    if SALVAR_SVG:
        fig.savefig(PASTA_FIGURAS / f"{nome_base}.svg", **common_kwargs)

    print("Figura salva:", PASTA_FIGURAS / nome_base)

    if MOSTRAR_FIGURAS:
        plt.show()
    else:
        plt.close(fig)


In [ ]:
# =============================================================================
# CÉLULA 17 — GERAR TODAS AS FIGURAS
# =============================================================================

def gerar_todas_as_figuras():
    """Gera todas as figuras nas variantes solicitadas."""
    variantes = []
    if GERAR_VERSAO_SEM_TITULOS_EIXOS:
        variantes.append(False)
    if GERAR_VERSAO_COM_TITULOS_EIXOS:
        variantes.append(True)

    if not variantes:
        raise ValueError("At least one export variant must be enabled.")

    for com_titulos_eixos in variantes:
        print("\n" + "=" * 80)
        print("Gerando versão:", sufixo_titulos_eixos(com_titulos_eixos))
        print("=" * 80)

        figura_01_anos_extremos_nacionais(com_titulos_eixos)
        figura_02_boxplot_macro(com_titulos_eixos)
        figura_03_boxplot_micro(com_titulos_eixos)
        figura_04_boxplot_solos(com_titulos_eixos)
        figura_05_barras_solo_macro(com_titulos_eixos)
        figura_06_heatmaps_solos(com_titulos_eixos)
        figura_07_paineis_solo_dentro_macro(com_titulos_eixos)

    print("\nConcluído.")
    print("Pasta de saída:", PASTA_FIGURAS)


In [ ]:
# =============================================================================
# CÉLULA 18 — EXECUÇÃO
# =============================================================================

gerar_todas_as_figuras()


In [ ]:
# =============================================================================
# CÉLULA 19 — CHECKLIST DE PUBLICAÇÃO
# =============================================================================
#
# Antes da submissão do manuscrito, faça uma inspeção visual de:
#
# 1. Legibilidade do texto no tamanho FINAL de impressão.
# 2. Ausência de rótulos de ticks ou legendas cortados.
# 3. Consistência das cores das macrorregiões entre mapas e gráficos.
# 4. Consistência das cores dos solos em todos os painéis baseados em solos.
# 5. Boxplots sem pontos internos de dados brutos/outliers.
# 6. Ausência de títulos das figuras.
# 7. Versão correta selecionada:
#       - no_axis_titles
#       - with_axis_titles
# 8. Preferir PDF/SVG para submissão vetorial.
# 9. Usar TIFF/PNG em 600 dpi apenas quando forem exigidos arquivos raster.
# 10. Não redimensionar figuras raster após a exportação, a menos que o periódico exija.
# =============================================================================
